**Midterm Analysis: cloud-Based Air Quality ML Pipeline**

This notebook answers the anaysis questions for air quality project which was executed in Google Cloud Colab Enterprise (Vertex AI).

**Midterm Analysis: cloud-Based Air Quality ML Pipeline**

This notebook answers the anaysis questions for air quality project which was executed in Google Cloud Colab Enterprise (Vertex AI).

**1. What challenges did you encounter when provisioning and executing machine learning workflows in a managed cloud environment (such as Colab Enterprise or Compute Engine) compared to local or standard free-tier notebooks?**

**Answer:**
Running this project in Colab Enterprise on Vertex AI was noticeably different from using the local notebook in Google Colab. The cloud environment added layers of setup, session management, and the cost tracking that is not readily visible locally.

**Setup:** Before writing code, I had to activate billing credits, select my project, enable the Vertex AI API, and create an e2-standard-4 runtime. My first code cell confirmed the environment.
Session management: Stopping the runtime saves credits but clears memory, so running cells out of order after a restart caused errors such as “NameError: name pd is not defined”. I learned to rerun the notebook from the top each session.

**Data loading:**  The uploaded AirQualityUCI.csv used semicolons as separators, commas as decimals, and a day-first date format with periods in the time (10/03/2004 18.00.00).  A month-first format string raised a ValueError until I used %d/%m/%Y %H.%M.%S.

**Cost awareness:** My billing report showed \$0.58 in usage, mostly from Vertex AI, fully covered by credits. Surprisingly, the largest charge ($0.34) was for SSD disk storage, which is billed even when the runtime is stopped.

**Submission:**  Because Colab Enterprise notebooks live in Google Cloud, I had to download the executed .ipynb  and push it to GitHub manually.
Overall, the cloud environment offered dedicated compute and integrated billing, but required more deliberate setup and cost monitoring.


**2. How did you identify and handle implicit missing values (such as -200 sentinel indicators) across the sensor columns, and how did your imputation strategy affect model stability?**


**Answers:**

The dataset records missing measurements as -200 instead of leaving them blank, a value that is impossible for pollutant concentrations, sensor responses, or humidity. Because -200 is a valid number, a standard missing-value check found no gaps, so I replaced every -200 with NaN and measured the share missing per column. NMHC(GT) was missing 90.2% of its values, the other reference analyzer columns were missing about 18%, and benzene, the five sensors, and the weather columns were each missing exactly 3.9%, suggesting those gaps occurred when the sensor device was offline.

I excluded NMHC(GT) because imputing it would mean inventing most of the column, and I dropped rows with a missing target rather than imputing them, leaving 7,674 rows for classification and 8,991 for regression. Remaining gaps in the numerical features were filled with the median, which suits skewed pollution data better than the mean, and temporal features used most-frequent imputation as a safeguard. This made the models stable: all ten classification models converged in 15 to 120 iterations, and Linear Regression reached an R squared of 0.9795. Since only 3.9% of feature values needed imputation, and the imputers learned only from training data, the median fill had little effect on results while preserving every usable row.


**3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps in your cloud pipeline.**






**Answers:**
The ColumnTransformer applies the right preprocessing to each feature type in one step: the eight numerical sensor and weather features receive median imputation and scaling, while the temporal features (Hour, Weekday, Month) receive most-frequent imputation and one-hot encoding, turning 11 inputs into 51 model features. The “Pipeline” then chains this preprocessing with the model, so a single .fit() call runs every step in order and .predict() applies the same transformations to new data. Building it as a reusable function let me run 10 classification and 13 regression models without duplicating code, and in the cloud, keeping everything in one object made it easy to rerun the workflow after each runtime restart.

The pipeline also prevents data leakage. The imputer, scaler, and encoder all learn values from the data, such as medians, means, and category lists, so fitting them on the full dataset would let test-set information influence training and inflate results. Because each pipeline was fit only on the training data, the test set was transformed using training statistics alone, just as new sensor readings would be in a real deployment. I also excluded all reference analyzer columns ending in (GT) from the features, since my targets were CO(GT) and C6H6(GT), and including related ground-truth pollutants would let the model read the answer rather than learn from the sensors.


**4. How did comparing StandardScaler versus MinMaxScaler impact your model training convergence, gradient descent optimization, and final evaluation metrics?**

**Answers:**
Scaling was essential for gradient descent because the raw features have very different ranges: sensor responses run into hundreds or thousands, while humidity values are around 1. Since gradient descent uses one learning rate for all features, unscaled data would let the large sensor values dominate the updates and slow convergence. Both scalers solved this, and all ten models converged. The effect on convergence depended on the learning rate setup: with SGD's default adaptive learning rate, MinMaxScaler stopped in fewer epochs (for example, 44 vs. 70 for hinge and 83 vs. 120 for modified Huber), because its narrow [0, 1] range produces smaller gradients that reach the stopping tolerance sooner. With a fixed learning rate of 0.01, however, StandardScaler converged nearly twice as fast (15 vs. 27 epochs), and softmax regression also converged faster with it (51 vs. 64 iterations).

StandardScaler produced higher accuracy in all five models, so MinMaxScaler's earlier stopping did not lead to better solutions. The best model, softmax regression with StandardScaler, reached 0.8619 accuracy and 0.9638 ROC-AUC, compared with 0.8528 and 0.9615 using MinMaxScaler. The gap was largest for the constant learning rate model (0.8443 vs. 0.8202) and hinge loss (0.8371 vs. 0.8169), although MinMaxScaler slightly outperformed on macro F1 and ROC-AUC for modified Huber. StandardScaler's zero-centered features suit gradient descent and L2 regularization, while MinMaxScaler is sensitive to pollution spikes that compress typical readings into a narrow range. Since all models trained in under a quarter of a second, the scaler's effect on accuracy mattered more than its effect on speed.


**5. How does configuring SGDClassifier differ from standard closed-form solvers, and what advantages do probability outputs and evaluation metrics provide when assessing classification performance?**

Standard solvers use the full dataset to find the solution: Linear Regression has an exact closed-form answer (the normal equation), and LogisticRegression uses the batch solver lbfgs, which takes a few precise steps using all training samples. SGDClassifier instead updates its weights one sample at a time, making many cheap but noisy steps across multiple passes. It also requires more configuration: the loss function determines the model type (log_loss for logistic regression, modified_huber for a smoothed classifier, and hinge for a linear SVM), along with the regularization strength, learning rate schedule, and stopping rules. In my results, softmax regression with lbfgs was the most accurate model (0.8619 accuracy), while the best SGD model, log_loss with a constant learning rate, reached 0.8443 in only 15 epochs and 0.06 seconds, nearly three times faster. SGD's speed, scalability, and support for online learning make it well suited to continuous sensor streams, though it is more sensitive to scaling and hyperparameters.

Probability outputs make the models more useful for air quality monitoring. Softmax regression and the SGD models using log_loss or modified_huber give a probability for each class, so an hour with a 45% chance of High pollution could trigger a precautionary alert even if Moderate is slightly more likely. Hinge loss cannot provide probabilities, which is why its ROC-AUC was missing. Using several metrics also gave a fuller picture: accuracy was meaningful because quantile binning created balanced classes; macro F1 weighted all three classes equally; ROC-AUC, which reached 0.9638 for my best model, showed how well the model ranked the correct class across all thresholds; and the confusion matrix revealed which classes were confused, with errors expected mainly around the Moderate class, which sits between the Low and High ranges.



**6. Based on your regularized regression models (Ridge versus Lasso), how did adjusting the regularization hyperparameter affect feature selection and coefficient shrinkage, and what operational environmental insights do these findings reveal?**

**Answers:**
Predicting benzene concentration C6H6(GT)  with all 51 features, the baseline Linear Regression achieved an R squared of 0.9795. Ridge (L2) shrank coefficients gradually but never to zero: its R squared stayed between 0.9795 and 0.9796 from alpha 0.001 to 10 and dipped only to 0.9766 at alpha 100, with all 51 features kept. Its coefficient paths showed the dominant PT08.S2(NMHC) coefficient shrinking from about 8.3 to 6.1, with weight shifting to correlated sensors such as PT08.S4(NO2). Lasso (L1) instead pushed coefficients to exactly zero, performing feature selection: it kept 47 features at alpha 0.001, 30 at 0.01, 6 at 0.1, and only 1 at alpha 1, before eliminating every feature at alphas of 10 and 100, where R squared fell to -0.0033 and the model simply predicted the average. The time features were removed first, followed by the weather variables and the other sensors.

These results offer clear operational insights. At alpha 1, Lasso kept only the PT08.S2(NMHC) sensor, which alone explained 94% of the variation in benzene (R squared = 0.9436), and with just 6 features at alpha 0.1, the model still reached an R squared of 0.9706. This suggests that a single low-cost hydrocarbon sensor can estimate benzene closely when expensive reference analyzers are offline, and that monitoring networks could prioritize maintaining and calibrating their most informative sensors. Temperature and humidity had negative coefficients at low alpha, consistent with metal oxide sensors being affected by weather, which supports calibrating sensors for environmental conditions. Because the sensors are correlated, however, these coefficients reflect associations rather than causes and should be validated with additional data.
